In [1]:
from IPython.display import HTML, display

def show(df, height=350):
    html = f'<div style="height:{height}px; overflow:auto; border:1px solid #ccc;">{df.to_html()}</div>'
    display(HTML(html))

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from dotenv import load_dotenv
from groq import Groq

load_dotenv('../.env')
print('GROQ_API_KEY loaded:', bool(os.environ.get('GROQ_API_KEY')))

GROQ_API_KEY loaded: True


1. Categories & System Prompt

In [3]:
CATEGORIES = [
    'Food & Drink',
    'Bills & Utilities',
    'Travel',
    'Groceries',
    'Health and Wellness',
    'Entertainment',
    'Shopping',
    'Personal',
]

print('Categories:', CATEGORIES)

Categories: ['Food & Drink', 'Bills & Utilities', 'Travel', 'Groceries', 'Health and Wellness', 'Entertainment', 'Shopping', 'Personal']


In [4]:
_SYSTEM_PROMPT = f"""You are a transaction categorizer. Given a numbered list of transaction descriptions, respond with the number and category for each, one per line, in the format "N. Category". Use only categories from this list:
{chr(10).join(f'- {c}' for c in CATEGORIES)}

Rules (apply in priority order â€” first matching rule wins):

FOOD & DRINK â€” restaurants, cafes, bars, fast food chains, takeout, food courts, canteens, bubble tea shops, coffee shops. The merchant must be a place where you consume food/drink on-site or get it delivered/takeaway. This includes non-English restaurant words: "Restauracja" (Polish), "Ristorante" (Italian), "Restaurante" (Spanish/Portuguese), "Restoran", etc.
Examples: McDonald's, Starbucks, Pret A Manger, Wendy's, Chick-fil-A, Chiptole, Nando's, Wagamama, Itsu, Shake Shack, Deliveroo, Uber Eats, Just Eat, any restaurant or cafÃ© name.

GROCERIES â€” supermarkets, food markets, grocery stores, Asian food stores, convenience stores. The merchant primarily sells unprepared food/household goods for home use.
Examples: Tesco, Sainsbury's, Waitrose, Lidl, Aldi, Whole Foods, Costco, Trader Joes, H-mart, and any market selling raw/packaged food.

TRAVEL â€” transportation and accommodation only. Flights, trains, buses, tubes, taxis, Uber (ride, not Eats), ferries, car rental, hotels, hostels, parking.
Examples: TfL, National Rail, Trainline, British Airways, easyJet, Uber (rides), Airbnb (accommodation), NCP parking, Delta Airlines,
NOT travel: activity/tour bookings, Amazon, general retail even if the name mentions a city.

ENTERTAINMENT â€” leisure activities, experiences, tours, ticketed events, activity booking platforms, cinemas, theatres, museums, streaming subscriptions, gaming.
Examples: GetYourGuide, Viator, Klook, Ticketmaster, Eventbrite, Vue Cinema, Netflix, Spotify, Steam, any tour or experience booking.

HEALTH AND WELLNESS â€” medical, dental, pharmacy, fitness, gym, yoga, mental health, personal care services (haircut, spa).
Examples: Boots (pharmacy products), GP surgery, dentist, gym membership, therapist, physical rehab

BILLS & UTILITIES â€” recurring service charges: internet, mobile phone plan, insurance, electricity, gas, water, council tax, streaming if it's a monthly subscription.
Examples: T-mobile, internet bills, council tax, Lycamobile (phone top-up/plan), utility providers.

SHOPPING â€” retail purchases of physical goods online or in-store, when not covered by Groceries, Health, or Entertainment.
Examples: Amazon, ASOS, H&M, Zara, Apple Store, electronics retailers. Default to Shopping when a merchant sells general goods and doesn't fit a more specific category. Amazon (including "AMAZON*" with order codes) is always Shopping unless the description explicitly says "Fresh" or "Pantry".

PERSONAL â€” financial transactions only: credit card payments, bank transfers, ATM withdrawals, direct debits to financial accounts.

Important notes:
- Ignore location suffixes in merchant names (e.g., "LONDON", "LONDONLND", "W1D", phone numbers, alphanumeric order IDs) â€” they are not categories.
- The same brand/merchant must always get the same category regardless of the suffix or transaction ID appended.
- When a merchant name is ambiguous, consider what the business primarily does.

Respond with ONLY numbered category lines, nothing else. Example output:
1. Food & Drink
2. Shopping
3. Travel
Every input number must appear exactly once."""

print(_SYSTEM_PROMPT[:300], '...')

You are a transaction categorizer. Given a numbered list of transaction descriptions, respond with the number and category for each, one per line, in the format "N. Category". Use only categories from this list:
- Food & Drink
- Bills & Utilities
- Travel
- Groceries
- Health and Wellness
- Entertai ...


2. Groq Client

In [5]:
_groq_client = None

def _get_client() -> Groq:
    global _groq_client
    if _groq_client is None:
        _groq_client = Groq(api_key=os.environ['GROQ_API_KEY'])
    return _groq_client

client = _get_client()
print('Groq client:', client)

Groq client: <groq.Groq object at 0x0000021CA542CFD0>


3. `_normalize` map raw LLM output to a valid category

In [6]:
def _normalize(result: str) -> str:
    for categories in CATEGORIES:
        if categories.lower() in result.lower():
            return categories
    print(f'WARNING: Unrecognized category: {result!r} --> defaulting to Personal')
    return 'Personal'

# Test cases
test_inputs = [
    'Food & Drink',
    'food & drink',
    'SHOPPING',
    'health and wellness',
    'Unknown Category XYZ',  # should fall back to Personal
]
for t in test_inputs:
    print(f'{t!r:30} -> {_normalize(t)}')

'Food & Drink'                 -> Food & Drink
'food & drink'                 -> Food & Drink
'SHOPPING'                     -> Shopping
'health and wellness'          -> Health and Wellness
'Unknown Category XYZ'         -> Personal


4. `_clean_description` — normalize descriptions before cache lookup

In [7]:
import re

_PHONE_RE        = re.compile(r'\b\d{3}[.\-\s]\d{3}[.\-\s]\d{4}\b')
_DOMAIN_RE       = re.compile(r'\.(?:COM|NET|ORG|IO)\b', re.I)
_STATE_NUM_RE    = re.compile(r'\b[A-Z]{2}\s+\d+\b')
_ORDER_CODE_RE   = re.compile(r'\b(?=[A-Z0-9]*[A-Z])(?=[A-Z0-9]*[0-9])[A-Z0-9]{5,}\b')
_TRAILING_NUM_RE = re.compile(r'(\s+\d+)+$')

def _clean_description(desc: str) -> str:
    desc = desc.replace('*', ' ')          # keep both sides: SQ *BREAD AHEAD -> SQ BREAD AHEAD
    desc = _PHONE_RE.sub('', desc)         # strip phone numbers
    desc = _DOMAIN_RE.sub('', desc)        # strip .COM / .NET domains
    desc = _STATE_NUM_RE.sub('', desc)     # strip "FL 9827", "CA 2453"
    desc = _ORDER_CODE_RE.sub('', desc)    # strip mixed alphanumeric order codes (e.g. AB12345)
    desc = _TRAILING_NUM_RE.sub('', desc)  # strip trailing digit sequences
    return ' '.join(desc.split())          # collapse whitespace

# Test cases
test_cases = [
    'AMAZON*AB12345',
    'SQ *BREAD AHEAD LTDLondon',
    'SUNDAY*Kricket CanarLondon',
    'SumUp *fresh meetcanning townGBR',
    'AIRBNB * HM3DXD8W3T AIRBNB.COM CA 2453 0504',
    'UNIVERSAL ORLANDO WEBSIT 407-224-4233 FL 9827 0504',
    'MCDONALDS LONDON',
    'NETFLIX.COM',
]
for raw in test_cases:
    print(f'{raw!r:55} -> {_clean_description(raw)!r}')


'AMAZON*AB12345'                                        -> 'AMAZON'
'SQ *BREAD AHEAD LTDLondon'                             -> 'SQ BREAD AHEAD LTDLondon'
'SUNDAY*Kricket CanarLondon'                            -> 'SUNDAY Kricket CanarLondon'
'SumUp *fresh meetcanning townGBR'                      -> 'SumUp fresh meetcanning townGBR'
'AIRBNB * HM3DXD8W3T AIRBNB.COM CA 2453 0504'           -> 'AIRBNB AIRBNB'
'UNIVERSAL ORLANDO WEBSIT 407-224-4233 FL 9827 0504'    -> 'UNIVERSAL ORLANDO WEBSIT'
'MCDONALDS LONDON'                                      -> 'MCDONALDS LONDON'
'NETFLIX.COM'                                           -> 'NETFLIX'


5. `_batch_categorize` send descriptions to Groq, get categories back

In [8]:
def _batch_categorize(descriptions: list[str], client: Groq) -> list[str]:
    user_message = '\n'.join(f'{i+1}. {desc}' for i, desc in enumerate(descriptions))

    try:
        response = client.chat.completions.create(
            model='llama-3.3-70b-versatile',
            messages=[
                {'role': 'system', 'content': _SYSTEM_PROMPT},
                {'role': 'user',   'content': user_message},
            ],
            temperature=0,
            max_tokens=20 * len(descriptions),
        )
    except Exception as exc:
        print(f'ERROR: Groq API call failed: {exc}')
        return ['Personal'] * len(descriptions)

    raw_lines = response.choices[0].message.content.strip().splitlines()
    print('Raw Groq response:')
    for line in raw_lines:
        print(' ', line)

    parsed: dict[int, str] = {}
    for line in raw_lines:
        line = line.strip()
        if not line:
            continue
        if '. ' in line:
            num_part, cat_part = line.split('. ', 1)
            if num_part.isdigit():
                parsed[int(num_part)] = _normalize(cat_part.strip())

    expected_indices = range(1, len(descriptions) + 1)
    missing = [idx for idx in expected_indices if idx not in parsed]
    if missing:
        print(f'WARNING: missing indices {missing}, defaulting to Personal')

    results = [parsed.get(idx, 'Personal') for idx in expected_indices]
    for idx, (desc, cat) in enumerate(zip(descriptions, results), 1):
        print(f'  {idx}. {desc} --> {cat}')
    return results

In [9]:
# Test with a small batch
test_descriptions = [
    'MCDONALDS LONDON',
    'AMAZON*AB12345',
    'TFL TRAVEL LONDON',
    'NETFLIX.COM',
    'TESCO SUPERSTORE',
]

results = _batch_categorize(test_descriptions, client)
print('\nFinal results:', results)

Raw Groq response:
  1. Food & Drink
  2. Shopping
  3. Travel
  4. Entertainment
  5. Groceries
  1. MCDONALDS LONDON --> Food & Drink
  2. AMAZON*AB12345 --> Shopping
  3. TFL TRAVEL LONDON --> Travel
  4. NETFLIX.COM --> Entertainment
  5. TESCO SUPERSTORE --> Groceries

Final results: ['Food & Drink', 'Shopping', 'Travel', 'Entertainment', 'Groceries']


6. Categorize_dataframe - Full pipeline with Supbabase Cache

In [10]:
import time
from database.connector import get_cached_categories_bulk, cache_categories_bulk

def categorize_dataframe(df):
    import time
    start_time = time.time()
    descriptions = df['description'].tolist()
    cleaned = [_clean_description(d) for d in descriptions]
    
    unique_cleaned = list(dict.fromkeys(cleaned))
    cache_results = get_cached_categories_bulk(unique_cleaned)
    uncached = [d for d in unique_cleaned if d not in cache_results]

    cache_hits = sum(1 for d in cleaned if d in cache_results)
    print(f'{len(descriptions)} transactions -> {len(unique_cleaned)} unique | {cache_hits} cached, {len(uncached)} need API')

    if uncached:
        _client = _get_client()
        chunk_size = 20
        chunks = [uncached[i:i+chunk_size] for i in range(0, len(uncached), chunk_size)]
        print(f'{len(uncached)} descriptions -> {len(chunks)} API call(s)')

        new_categories = []
        for i, chunk in enumerate(chunks, 1):
            batch = _batch_categorize(chunk, _client)
            new_categories.extend(batch)
            new_mappings = dict(zip(chunk, batch))
            cache_results.update(new_mappings)
            cache_categories_bulk(new_mappings)
            print(f'  Cached batch {i}/{len(chunks)}')
        print(f'Added {len(new_categories)} new descriptions to cache')

    categories = [cache_results[c] for c in cleaned]
    print(categories)
    elapsed = time.time() - start_time
    print(f'Categorization finished in {elapsed:.1f}s')

    df = df.copy()
    df['cleaned_description'] = cleaned
    df['category'] = categories
    return df

In [11]:
from pipeline.pdf_parser import parse_pdf

# PDF_PATH = '../data/BOFA_072025_0504.pdf'
# PDF_PATH = '../data/Capital_One_102025_2952.pdf'
PDF_PATH = "../data/Chase_Sapphire_20251217-1333.pdf"

parsed = parse_pdf(PDF_PATH)
transactions_df = parsed['transactions']

print(f"Card: {parsed['card_name']}  ****{parsed['last_four']}")
print(f"Period: {parsed['period_start']} to {parsed['period_end']}")
print(f"{len(transactions_df)} transactions")
transactions_df

Card: Chase Sapphire Preferred  ****1333
Period: 2025-11-18 to 2025-12-17
87 transactions


,trans_date,description,amount1,amount2
0,11/20,Payment Thank You-Mobile,-859.67,None
1,11/27,Payment Thank You-Mobile,-700.00,None
2,11/28,HM Hennes Mauritz UK L London,-46.18,None
3,12/04,Payment Thank You-Mobile,-388.72,None
4,12/14,Payment Thank You-Mobile,-726.34,None
...,...,...,...,...
82,12/12,TFL TRAVEL CH TFL.GOV.UK/CP,7.77,None
83,12/13,TFL TRAVEL CH TFL.GOV.UK/CP,9.52,None
84,12/14,Amazon Fresh amazon.co.uk,3.79,None
85,12/16,GETYOURGUIDE TICKETS 855-957-1272 NY,29.15,None


In [12]:
categorized_df = categorize_dataframe(transactions_df)
show(categorized_df)

87 transactions -> 50 unique | 0 cached, 50 need API
50 descriptions -> 3 API call(s)
Raw Groq response:
  1. Personal
  2. Shopping
  3. Groceries
  4. Travel
  5. Travel
  6. Travel
  7. Groceries
  8. Groceries
  9. Food & Drink
  10. Shopping
  11. Travel
  12. Food & Drink
  13. Food & Drink
  14. Shopping
  15. Shopping
  16. Health and Wellness
  17. Food & Drink
  18. Travel
  19. Shopping
  20. Food & Drink
  1. Payment Thank You-Mobile --> Personal
  2. HM Hennes Mauritz UK L London --> Shopping
  3. TIAN TIAN MARKET - CANARY LONDON --> Groceries
  4. CL Chase Travel TRIPCHRG VA --> Travel
  5. VENICE TRANSPORT VENEZIA --> Travel
  6. TFL TRAVEL CH TFL.GOV.UK/CP --> Travel
  7. ASDA SUPERSTORE ISLE OF DOGS --> Groceries
  8. SAINSBURYS S/MKTS THE CITY -MAN --> Groceries
  9. SQ PEPPER STREET TAVERN London --> Food & Drink
  10. GlobalE /Goodai Global In New York NY --> Shopping
  11. UBR PENDING.UBER AMSTERDAM --> Travel
  12. PRET A MANGER LONDON --> Food & Drink
  13. KARAK

,trans_date,description,amount1,amount2,cleaned_description,category
0,11/20,Payment Thank You-Mobile,-859.67,None,Payment Thank You-Mobile,Personal
1,11/27,Payment Thank You-Mobile,-700.00,None,Payment Thank You-Mobile,Personal
2,11/28,HM Hennes Mauritz UK L London,-46.18,None,HM Hennes Mauritz UK L London,Shopping
3,12/04,Payment Thank You-Mobile,-388.72,None,Payment Thank You-Mobile,Personal
4,12/14,Payment Thank You-Mobile,-726.34,None,Payment Thank You-Mobile,Personal
5,11/17,TIAN TIAN MARKET - CANARY LONDON,53.06,None,TIAN TIAN MARKET - CANARY LONDON,Groceries
6,11/17,CL *Chase Travel TRIPCHRG.COM VA,42.81,None,CL Chase Travel TRIPCHRG VA,Travel
7,11/16,VENICE TRANSPORT VENEZIA,1.75,None,VENICE TRANSPORT VENEZIA,Travel
8,11/16,TFL TRAVEL CH TFL.GOV.UK/CP,2.31,None,TFL TRAVEL CH TFL.GOV.UK/CP,Travel
9,11/17,TFL TRAVEL CH TFL.GOV.UK/CP,9.24,None,TFL TRAVEL CH TFL.GOV.UK/CP,Travel
